# Gemma 3-4B, 1-2B LoRA Fine-Tuning


## 라이브러리 설치

In [1]:
try:
    import google.colab
    inColab = True
except ImportError:
    inColab = False

In [2]:
if inColab == True:
    !pip install -U pandas==2.2.2 numpy==2.0.2 scipy==1.14.1 accelerate==1.6.0 peft==0.15.2 bitsandbytes==0.45.5 transformers==4.51.3 trl==0.16.1 datasets==3.5.0 tensorboard==2.19.0

## 라이브러리 선언

In [3]:
import os
import random
import numpy as np
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datetime import datetime

#  수정 포인트: 재현성(Seed 고정)
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

GPU: NVIDIA RTX 2000 Ada Generation


In [4]:
from dotenv import load_dotenv
load_dotenv()

import huggingface_hub
huggingface_hub.login(os.getenv('HF_TOKEN'))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 1. 모델 및 데이터셋 설정

In [5]:
# base_model = "google/gemma-2b-it"
# base_model = "google/gemma-3-4b-it"
# base_model = "familicare/elysium_general_medical"
base_model = "google/gemma-3-4b-it"
dataset_name = os.getenv('HF_DATASET_REPO', 'yunhwa/law_instruct')

# 데이터 로드
ds = load_dataset(dataset_name, split="train")

# 비율로 분할 (80% Train, 20% Eval)
# 데이터가 늘어나도 자동으로 비율이 유지됩니다.
ds = ds.train_test_split(test_size=0.2, seed=SEED)
train_ds = ds["train"]
eval_ds  = ds["test"]

print(f"Total: {len(ds['train']) + len(ds['test'])}")
print(f"Train size: {len(train_ds)} (80%)")
print(f"Eval  size: {len(eval_ds)} (20%)")

Total: 253207
Train size: 202565 (80%)
Eval  size: 50642 (20%)


In [6]:
# train_ds_filtered = train_ds.filter(lambda x: x["input"] == "내과")
# train_ds_filtered = train_ds_filtered.select(range(min(100, len(train_ds_filtered))))

In [7]:
import pandas as pd
from datasets import Dataset

ds_full = load_dataset(dataset_name, split='train')

# 도메인(input)별 최대 100개 균등 샘플링 후 전체 셔플
df = ds_full.to_pandas()
sampled_df = (
    df.groupby('input', group_keys=False)
      .apply(lambda g: g.sample(min(100, len(g)), random_state=SEED))
      .reset_index(drop=True)
      .sample(frac=1, random_state=SEED)
)

train_ds = Dataset.from_pandas(sampled_df, preserve_index=False)

print(f'샘플링 후 학습 데이터: {len(train_ds)}개')
print(sampled_df['input'].value_counts().to_string())


샘플링 후 학습 데이터: 400개
input
행정법       100
민사법       100
형사법       100
지식재산권법    100


C:\Users\SMT21\AppData\Local\Temp\ipykernel_23620\2333986078.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(100, len(g)), random_state=SEED))


In [8]:
train_ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 400
})

## 2. 토크나이저 설정 및 채팅 템플릿 적용

In [9]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

# Gemma3는 token_type_ids를 입력으로 받지 않으므로 명시적으로 제외
tokenizer.model_input_names = ['input_ids', 'attention_mask']


In [10]:
def to_chat_text(example):
    instruction = example["instruction"].strip()
    user_input  = example["input"].strip()
    output      = example["output"].strip()

    user_msg = (
        f"{instruction}\n\n[입력]\n{user_input}" if instruction and user_input
        else instruction or user_input
    )

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": output},
    ]

    try:
        example["text"] = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    except Exception:
        example["text"] = (
            "<start_of_turn>user\n"
            f"{user_msg}\n"
            "<end_of_turn>\n"
            "<start_of_turn>model\n"
            f"{output}\n"
            "<end_of_turn>"
        )

    return example

In [11]:
train_ds = train_ds.map(
    to_chat_text,
    remove_columns=["instruction", "input", "output"]
)

eval_ds = eval_ds.map(
    to_chat_text,
    remove_columns=["instruction", "input", "output"]
)


# 변환된 데이터 확인
print("Sample Text:")
print(train_ds[0]["text"][:500])

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Sample Text:
<bos><start_of_turn>user
상표권자의 '상당한 주의'는 어떤 의미인가요?

[입력]
행정법<end_of_turn>
<start_of_turn>model
상표법 제73조 제1항 제8호 단서에서 규정하는 상표권자의 '상당한 주의'란 단순히 오인이나 혼동을 피하도록 주의나 경고를 하는 것만으로는 충분하지 않으며, 상표의 사용 실태를 실질적이고 정기적으로 감독하거나 보고를 받는 방법을 통해 사용권자를 실질적으로 통제하고 있는 관계가 유지되어야 한다는 의미입니다. 이는 상표권자가 사용권자의 상표 사용을 관리하고 지배 아래 두고 있는 상태를 유지하고 있는 것이 인정되어야 합니다. 참조 법령은 상표법 제73조 제1항 제8호입니다.<end_of_turn>



In [12]:
eval_ds

Dataset({
    features: ['text'],
    num_rows: 50642
})

In [13]:
# train_ds = train_ds.filter(lambda x: x["input"] == "내과")
# train_ds = train_ds.select(range(100))
# eval_ds=eval_ds.select(range(20))

In [14]:
train_ds[1]

{'text': '<bos><start_of_turn>user\n통합 발행된 국고채권을 다시 발행할 수 있나요?\n\n[입력]\n행정법<end_of_turn>\n<start_of_turn>model\n국채법 제7조 제2항에 따라 기획재정부장관은 국채시장의 안정적 관리 등을 위하여 필요하다고 인정하는 경우, 통합 발행된 국고채권을 일정한 기간이 끝난 후에도 다시 발행할 수 있습니다.<end_of_turn>\n'}

## 3. 환경 및 최적화 설정 (4-bit 양자화)

In [15]:
# 현재 사용 중인 GPU의 주요 아키텍처 버전을 반환 8버전 이상 시 bfloat16 활용
# NVIDIA Ampere 아키텍처 이상 시에만 처리
if torch.cuda.get_device_capability()[0] >= 8:
    # 고속 attention 메커니즘을 구현하는 라이브러리
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16


# BitsAndBytesConfig 객체활용 양자화 설정
quant_config = BitsAndBytesConfig(
    # 모델을 4비트 양자화하여 로드할지 여부 결정
    load_in_4bit=True,
    # 양자화 방법 (nf4: Non-Uniform Quantization, "nf4","fp16 등))
    bnb_4bit_quant_type="nf4",
    # (4비트 양자화 시 사용할 데이터 타입, "torch.float16, bfloat16, float32 등)
    bnb_4bit_compute_dtype=torch_dtype,
    # 이중 양자화 사용여부 (이중 양자화는 양자화 과정에서 정밀도 높이기 위해 활용, 대신 더 연산은 복잡)
    bnb_4bit_use_double_quant=False
)

## 4. 베이스모델 불러오기

In [16]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U accelerate


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
model = AutoModelForCausalLM.from_pretrained(
    # 불러올 모델 정의
    base_model,
    # 모델 양자화 설정값
    quantization_config=quant_config,
    # 모델의 레이어를 할당할 장치 ("":0 -> 전체 모델을 GPU 0에 할당, "auto"는 알아서, "{"layer_0":0, ... 형태로 레이어별 할당 가능)
    device_map="auto",
)

# 학습 시 캐시 사용 중지 (추론 시엔 켜야 함)
# 학습에 불필요한 캐시·병렬 프리트레인 설정을 꺼서 안정적으로 학습하겠다
# 캐시는 추론용이라, 학습 중엔 메모리만 잡아먹고 오히려 문제
model.config.use_cache = False
# Tensor Parallelism 분할을 끈 상태
#일부 LLaMA 계열에서 값이 1이 아니면 loss 깨짐/느려짐 이슈가 있어, 파인튜닝 시 보통 1로 고정
model.config.pretraining_tp = 1

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

## 5. LoRA 및 Trainer 설정

In [18]:
# LoRA 설정 (어댑터 학습 파트)
peft_params = LoraConfig(    
    # LoRA 랭크(rank). 값이 클수록 "학습 가능한 파라미터"가 늘어서 성능 잠재력은 올라가지만
    # VRAM/학습시간도 조금 증가. (보통 8, 16, 32 자주 사용)
    r=16,

    # LoRA 스케일링 계수. 대체로 r과 비슷하거나 2배 정도로 두는 경우가 많음.
    # 너무 크면 불안정, 너무 작으면 학습이 약해질 수 있음.
    lora_alpha=32,

    # LoRA 레이어에 dropout. 과적합 방지용(데이터가 적을수록 유리).
    # 0.0 ~ 0.1 사이에서 많이 씀.
    lora_dropout=0.05,

    # LoRA는 보통 bias는 학습하지 않음(메모리/일관성 측면에서 깔끔).
    # "all" 같은 옵션도 있지만 대개 none으로 충분.
    bias="none",

    # 원인-결과(next token prediction) 방식의 언어모델 학습(SFT)임을 의미.
    task_type="CAUSAL_LM",

    # "어떤 레이어에 LoRA를 꽂을지" 지정.
    # Transformer의 attention/FFN 핵심 projection들에만 어댑터를 붙여서
    # 전체 모델을 다 학습하지 않고도 성능을 올리는 방식.
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [19]:
# SFTTrainer 학습 설정
sft_args = SFTConfig(
    output_dir="./results",

    # (A) 학습량 / 시간 제어
    # 학습 데이터 전체를 몇 번(epoch) 반복할지
    # 예: 데이터가 약 10,000개라면 1~3 epoch부터 시작 추천
    num_train_epochs=3,
    # GPU 1장(L4 24GB) 기준, 한 번에 올릴 샘플 수
    # OOM 발생 시 가장 먼저 줄일 값
    per_device_train_batch_size=1,

    # 평가 시 배치 크기 (eval도 VRAM 사용)
    per_device_eval_batch_size=1,

    # gradient 누적 스텝 수
    # 유효 배치 크기 = batch_size × gradient_accumulation_steps
    # 예: 1 × 4 = 4
    gradient_accumulation_steps=4,

    # (B) 입력 길이 / 속도 / 메모
    # 최대 토큰 길이
    # 512 → 256으로 줄이면 속도·VRAM 크게 절약됨
    max_length=256,

    # 짧은 샘플들을 하나의 시퀀스로 묶어 학습 효율 증가
    packing=True,

    # 길이가 비슷한 샘플끼리 묶어 패딩 최소화
    # group_by_length=True,

    # 학습에 사용할 텍스트 컬럼명
    # train_ds.map()에서 생성한 "text" 컬럼
    dataset_text_field="text",

    # (C) 최적화 하이퍼파라미터
    # LoRA SFT에서 자주 쓰이는 학습률
    learning_rate=2e-4,

    # 가중치 감쇠 (과적합 완화)
    weight_decay=0.001,

    # gradient clipping (폭주 방지)
    max_grad_norm=0.3,

    # 워밍업 비율 (초반 학습 안정화)
    warmup_ratio=0.03,

    # 워밍업 이후 LR 고정
    lr_scheduler_type="constant",

    # (D) 로그 / 평가 / 저장
    # 몇 step마다 로그 출력
    logging_steps=1000,

    # TensorBoard 로깅
    report_to="tensorboard",

    # 평가 비활성화 (속도 우선)
    eval_strategy="no",
    eval_steps=1000,

    # (ref_added) Evaluation 설정 추가
    # eval_strategy="steps",      # step 단위 평가
    # eval_steps=1000,            # 1000 step마다 평가

    # # best model 자동 로드
    # load_best_model_at_end=True,
    # metric_for_best_model="loss",
    # greater_is_better=False,
    
    # 체크포인트 저장 주기
    save_strategy="steps",
    save_steps=1000,

    # 최대 체크포인트 개수 제한
    save_total_limit=3,

    # (E) 혼합정밀도 / 재현성
    # GPU 타입에 따라 fp16 / bf16 사용
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),

    # 시드 고정
    seed=SEED,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# 모델 훈련

In [20]:
trainer = SFTTrainer(
    # 학습할 모델
    model=model,
    # 모델 학습에 사용할 데이터셋
    train_dataset=train_ds,
    eval_dataset=None,        # <- 평가를 쓰면 넣고, 안 쓰면 None 가능 대신 상단 eval_strategy="no"
    # Peft (파라미터 효율적 미세 조정) 설정 정의
    peft_config=peft_params,
    #모델에 함께 사용할 토크나이저
    processing_class=tokenizer,  # TRL 최신 방식: tokenizer 전달
    # 훈련 파라미터 설정
    args=sft_args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Converting train dataset to ChatML:   0%|          | 0/400 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

In [21]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


ValueError: `token_type_ids` is required as a model input when training

In [23]:
# %load_ext tensorboard
# %tensorboard --logdir=./results/runs

## 7. 어댑터 저장 (학습된 LoRA Adapter만 저장하여 용량을 줄임)

In [24]:
now_str = datetime.now().strftime("%Y_%m_%d_%H")
preFix = base_model.split("/")[1]
savePath = f"/content/gdrive/MyDrive/Colab Notebooks/models/{preFix}_{now_str}"

if inColab == True:
    savePath = f"/content/gdrive/MyDrive/Colab Notebooks/models/{preFix}_{now_str}"
    trainer.save_model(savePath)
else:
    savePath = f"./models/{preFix}_{now_str}"
    trainer.save_model(savePath)

# 어댑터 및 토크나이저 저장
trainer.model.save_pretrained(savePath)
tokenizer.save_pretrained(savePath)
# trainer.save_model(savePath)
print("저장 완료:", savePath)

저장 완료: ./models/gemma-3-4b-it_2026_04_10_14


## 참고. 추론 테스트 (현재 세션)

In [22]:
train_ds[10]

{'text': '<bos><start_of_turn>user\ntokenizer 역할?\n\n[입력]\nAI 초보자에게 쉽게 설명해줘<end_of_turn>\n<start_of_turn>model\n문자열을 숫자로 바꿔서 AI가 이해하게 해(max_length=512)<end_of_turn>\n'}

In [23]:
def infer_gemma(
    question,
    input_text=None,
    system=None,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
):
    model.eval()

    # -----------------
    # 메시지 구성
    # -----------------
    messages = []
    if system:
        messages.append({"role": "system", "content": system})

    user_msg = question if not input_text else f"{question}\n\n[입력]\n{input_text}"
    messages.append({"role": "user", "content": user_msg})

    # -----------------
    # 프롬프트 생성
    # -----------------
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = (
            "<start_of_turn>user\n"
            f"{user_msg}\n"
            "<end_of_turn>\n"
            "<start_of_turn>model\n"
        )

    # -----------------
    # 토크나이즈 + vocab 안전 체크
    # -----------------
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

    if inputs["input_ids"].max() >= model.get_input_embeddings().num_embeddings:
        raise ValueError("token id out of range (tokenizer/model vocab mismatch)")

    inputs = inputs.to(model.device)
    
    # -----------------
    # dtype / autocast
    # -----------------
    model_dtype = next(model.parameters()).dtype
    amp_dtype = model_dtype if model_dtype in (torch.float16, torch.bfloat16) else torch.float16

    # -----------------
    # 생성
    # -----------------
    with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,  # Gemma + FP16 안정화
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [30]:
# -----------------
# 예시 입력 구성
# -----------------
system_msg = "보기 중 가장 옳은 답을 하나 고릅니다."

question = (
    "Hydroxychloroquine 사용 시 발생할 수 있는 대표적인 부작용은 무엇인가?  \n1) 혈당 감소  \n2) 면역 억제  \n3) QT 간격 연장  \n4) 혈압 상승"
)

input_text = "마취통증의학과"

# -----------------
# 추론 호출
# -----------------
answer = infer_gemma(
    system=system_msg,
    question=question,
    input_text=input_text,
    max_new_tokens=64,
    temperature=0.1,
    top_p=0.9,
)

print(answer)

You have set `use_cache` to `False`, but cache_implementation is set to hybrid. cache_implementation will have no effect.


user
보기 중 가장 옳은 답을 하나 고릅니다.

Hydroxychloroquine 사용 시 발생할 수 있는 대표적인 부작용은 무엇인가?  
1) 혈당 감소  
2) 면역 억제  
3) QT 간격 연장  
4) 혈압 상승

[입력]
마취통증의학과
model
3) QT 간격 연장



In [24]:
### 전문 문제

In [25]:
# -----------------
# 예시 입력 구성
# -----------------
system_msg = "문제에 대한 답을 제시하세요"

question = (
    "60세 남성이 만성 요통과 다리 저림을 호소하며 내원했다. MRI에서 요추 추간판 탈출증이 확인되었다. 이 환자의 초기 치료 방침을 설명하시오."
)

input_text = ""

# -----------------
# 추론 호출
# -----------------
answer = infer_gemma(
    system=system_msg,
    question=question,
    input_text=input_text,
    max_new_tokens=64,
    temperature=0.1,
    top_p=0.9,
)

print(answer)

You have set `use_cache` to `False`, but cache_implementation is set to hybrid. cache_implementation will have no effect.


user
문제에 대한 답을 제시하세요

60세 남성이 만성 요통과 다리 저림을 호소하며 내원했다. MRI에서 요추 추간판 탈출증이 확인되었다. 이 환자의 초기 치료 방침을 설명하시오.
model
이 환자는 만성 요통과 다리 저림 증상을 호소하며, MRI에서 요추 추간판 탈출증이 확인되었으므로, 초기 치료로는 비수술적 방법을 우선적으로 고려해야 한다. 비스테로이드성 항염증제(NSAIDs)를 사용하여 염
